In [1]:
from datasets import load_dataset
posts = load_dataset("trl-lib/tldr")

ModuleNotFoundError: No module named 'datasets'

In [ ]:
def getSubredditFromPost(post):
    return  post['prompt'].split('\n')[0][11:];

counts = {}
subreddits = []
for post in posts['train']:
    subreddit = getSubredditFromPost(post)
    if subreddit not in subreddits:
        subreddits.append(subreddit)
    if(subreddit in counts):
        counts[subreddit] = counts[subreddit] + 1
    else:
        counts[subreddit] = 1

amount_of_posts_to_evaluate = min(counts.values())


In [6]:
def getSubredditFromPost(post):
    return  post['prompt'].split('\n')[0][11:];

counts = {}
subreddits = []
for post in posts['train']:
    subreddit = getSubredditFromPost(post)
    if subreddit not in subreddits:
        subreddits.append(subreddit)
    if(subreddit in counts):
        counts[subreddit] = counts[subreddit] + 1
    else:
        counts[subreddit] = 1

for post in posts['test']:
    subreddit = getSubredditFromPost(post)
    if subreddit not in subreddits:
        subreddits.append(subreddit)
    if(subreddit in counts):
        counts[subreddit] = counts[subreddit] + 1
    else:
        counts[subreddit] = 1

for post in posts['validation']:
    subreddit = getSubredditFromPost(post)
    if subreddit not in subreddits:
        subreddits.append(subreddit)
    if(subreddit in counts):
        counts[subreddit] = counts[subreddit] + 1
    else:
        counts[subreddit] = 1

amount_of_posts_to_evaluate = min(counts.values())
counts

{'r/relationships': 70325,
 'r/loseit': 1624,
 'r/personalfinance': 2583,
 'r/offmychest': 1731,
 'r/relationship_advice': 9641,
 'r/dating_advice': 3170,
 'r/dogs': 705,
 'r/legaladvice': 2200,
 'r/AskReddit': 17224,
 'r/running': 638,
 'r/tifu': 8523,
 'r/needadvice': 580,
 'r/cats': 364,
 'r/Advice': 2320,
 'r/BreakUps': 946,
 'r/pettyrevenge': 610,
 'r/self': 1179,
 'r/GetMotivated': 193,
 'r/Parenting': 483,
 'r/weddingplanning': 476,
 'r/college': 309,
 'r/jobs': 1191,
 'r/Dogtraining': 404,
 'r/Pets': 410,
 'r/Cooking': 121,
 'r/askwomenadvice': 762,
 'r/AskDocs': 323,
 'r/travel': 504,
 'r/books': 183}

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1

pipe = pipeline("summarization", model="google/pegasus-xsum", device=0)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
# (C) Andrew Foland, Sonnetiq, 2024; license granted under Apache 2.0
import torch
from transformers import AutoModel, AutoTokenizer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity(vector1, vector2):
    # Calculate cosine similarity between two vectors
    return cosine_similarity([vector1], [vector2])[0][0]

def embed_string(text, model, tokenizer):
    input_ids = tokenizer.encode(text, return_tensors='pt', max_length=512, truncation=True)
    with torch.no_grad():
        embedding = model(input_ids).last_hidden_state.mean(dim=1).squeeze().tolist()
    return embedding

def NOIR(text, summary, embedding_model = "sentence-transformers/all-MiniLM-L6-v2"):
    model = AutoModel.from_pretrained(embedding_model)
    tokenizer = AutoTokenizer.from_pretrained(embedding_model)

    text_embedding = embed_string(text, model, tokenizer)
    summary_embedding = embed_string(summary, model, tokenizer)
    D = calculate_cosine_similarity(text_embedding, summary_embedding)

    text_length = len(tokenizer.encode(text))
    summary_length = len(tokenizer.encode(summary))
    k = summary_length / text_length

    sque_metric = np.log(k) / np.log(D)
    return sque_metric

In [ ]:
import time
import evaluate

noirScores = {}
bleurtScores = {}
rougeScores = {}
counter = 0

rouge = evaluate.load("rouge")
bleurt = evaluate.load("bleurt", module_type="metric")

for subreddit in subreddits:
  noirScores[subreddit] = []
  sub_dataset = posts.filter(lambda post : getSubredditFromPost(post) == subreddit)
  start = time.time()
  completions = []
  labels = []
  for post in sub_dataset['train'].select(range(amount_of_posts_to_evaluate)):
    generatedText = pipe(post['prompt'])
    
    noirScore = NOIR(post['prompt'], generatedText[0]['summary_text'])
    noirScores[subreddit].append(noirScore)

    completions.append(generatedText)
    labels.append(post['completion'])

  bleurtScore = bleurt.compute(predictions=completions, references=labels)
  bleurtScores[subreddit] = bleurtScore['scores']

  rougeScore = rouge.compute(predictions=completions, references=labels)
  rougeScores[subreddit] = rougeScore['scores']

  end = time.time()
  print("Time: " + str(end-start))



Filter:   0%|          | 0/116722 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6447 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6553 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

...

Token indices sequence length is longer than the specified maximum sequence length for this model (525 > 512). Running this sequence through the model will result in indexing errors


Time: 1.1918177604675293
576
Time: 1.1741447448730469
577
Time: 0.8574943542480469
578
Time: 0.7657420635223389
579
Time: 1.0337955951690674
580
Time: 1.247558832168579
581
Time: 1.1786003112792969
582
Time: 1.1784000396728516
583
Time: 1.4515249729156494
584
Time: 0.8442957401275635
585
Time: 1.1673483848571777
586
Time: 1.3844451904296875
587
Time: 1.227541208267212
588
Time: 0.8078396320343018
589
Time: 0.9480020999908447
590
Time: 1.1301977634429932
591
Time: 1.178452491760254
592
Time: 1.1936190128326416
593
Time: 0.9206545352935791
594
Time: 0.7866315841674805
595
Time: 1.083751916885376
596
Time: 1.378176212310791
597
Time: 1.0361607074737549
598
Time: 1.0671641826629639
599
Time: 1.2734355926513672
600
Time: 1.4319653511047363
601
Time: 1.2903993129730225
602


Token indices sequence length is longer than the specified maximum sequence length for this model (519 > 512). Running this sequence through the model will result in indexing errors


Time: 0.8500440120697021
603


AcceleratorError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import matplotlib.pyplot as plt

# Remove NaN values from the scores
cleaned_scores = my_dictionary = {k: ~np.isnan(v) for k, v in scores.items()}
fig = plt.figure(figsize =(10, 7))

plt.boxplot(scores)
plt.show()

In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

2.8.0+cu126
12.6
CUDA available: False
No GPU


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 500: named symbol not found (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
